# C2O Research Notebook: Model Run And Tests

This notebook is the code-section research implementation for the C2O coursework. It runs the research chain from raw coursework data: point-in-time panel construction, baseline model, promoted Phase 2 model, final portfolio, robustness checks, submission-file audit, and tests.

Precomputed `outputs/*.csv` files are not used to produce the research results below. They are read only near the end to audit that the submitted artefacts match the notebook run.


## Research Plan

1. Fix the coursework assumptions and verify raw input files.
2. Build the point-in-time stock-day panel from raw parquet data.
3. Audit the return identity and equal-weight close/open/close decomposition.
4. Train the baseline walk-forward model and backtest the 250M portfolio.
5. Train the promoted Phase 2 expanding-window model and backtest the final 250M portfolio.
6. Run validation/holdout, ablation, capacity, cost, and submission-file checks.
7. Run the submitted pytest suite directly from the notebook.


In [1]:
from pathlib import Path
import os
import platform
import subprocess
import sys
import time

import numpy as np
import pandas as pd


def find_repo_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / 'src' / 'c2o_strategy').exists() and (candidate / 'tests').exists():
            return candidate
    raise RuntimeError('Could not locate 03_Code_Repository root.')

ROOT = find_repo_root(Path.cwd())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

np.random.seed(42)
pd.set_option('display.max_columns', 40)
pd.set_option('display.float_format', lambda x: f'{x:,.6f}')

print(f'Notebook root: {ROOT}')
print(f'Python: {sys.executable}')
print(f'Platform: {platform.platform()}')


Notebook root: /Users/ziyunjameschen/Downloads/IC_本地/ML for Fin HW/Final_Submission_Package_607/03_Code_Repository
Python: /opt/anaconda3/bin/python
Platform: macOS-26.5.1-arm64-arm-64bit-Mach-O


In [2]:
from c2o_strategy.config import StrategyConfig
from c2o_strategy.data import prepare_data
from c2o_strategy.experiments import (
    ExperimentSpec,
    add_walk_forward_alpha_return,
    ic_summary,
    run_experiment_backtest,
    score_panel_with_target,
    slim_panel,
    summarise_experiment,
)
from c2o_strategy.features import (
    add_cross_sectional_ranks,
    add_point_in_time_features,
    apply_eligibility_filters,
    build_eligibility_summary,
)
from c2o_strategy.final_strategy import (
    FEATURE_GROUPS,
    final_backtest_with_positions,
    final_phase2_spec,
    performance_row,
)
from c2o_strategy.phase2 import (
    PERIODS,
    ic_for_period,
    run_phase2_backtest,
    score_panel_phase2,
    summarise_period,
)
from c2o_strategy.reporting import compute_equal_weight_decomposition

config = StrategyConfig(data_dir=ROOT / 'data', output_dir=ROOT / 'outputs')
AUM = 250_000_000.0

pd.Series({
    'start_date': config.start_date,
    'cutoff': config.cutoff,
    'development_cutoff': config.development_cutoff,
    'universe_size': config.universe_size,
    'participation_cap': config.participation_cap,
    'basket_fraction': config.basket_fraction,
    'AUM_used_for_research_run': AUM,
})


start_date                          2010-01-01
cutoff                              2024-12-31
development_cutoff                  2024-12-31
universe_size                             1000
participation_cap                     0.050000
basket_fraction                       0.030000
AUM_used_for_research_run   250,000,000.000000
dtype: object

## 1. Raw Inputs And Fixed Assumptions

The notebook starts from the raw coursework files in `data/`. The cleaned submission keeps generated output files for audit and tests, but model training below reads the raw parquet/csv input data directly.


In [3]:
expected_data_files = [
    'prices.parquet', 'sp500_tr.parquet', 'sp500_constituents.parquet',
    'sp400_constituents.parquet', 'all_data.parquet', 'cheapness_scores.parquet',
    'earnings_calendar.parquet', 'earnings_transfo.parquet', 'short_interest_transfo.parquet',
    'piotrosky.parquet', 'gics_info.parquet', 'regime.parquet',
    'rolling_scores_upgrade.csv', 'rolling_scores_downgrade.csv',
]

file_check = pd.DataFrame({
    'file': expected_data_files,
    'exists': [(config.data_dir / name).exists() for name in expected_data_files],
    'size_mb': [
        (config.data_dir / name).stat().st_size / 1024**2 if (config.data_dir / name).exists() else np.nan
        for name in expected_data_files
    ],
})
assert file_check['exists'].all(), file_check.loc[~file_check['exists'], 'file'].tolist()
file_check


,file,exists,size_mb
0,prices.parquet,True,238.194407
1,sp500_tr.parquet,True,0.263647
2,sp500_constituents.parquet,True,0.020534
3,sp400_constituents.parquet,True,0.011790
4,all_data.parquet,True,241.975775
5,cheapness_scores.parquet,True,312.805212
6,earnings_calendar.parquet,True,0.365951
7,earnings_transfo.parquet,True,97.750225
8,short_interest_transfo.parquet,True,14.070617
9,piotrosky.parquet,True,0.145398


## 2. Build The Point-In-Time Research Panel

This is the heavy research cell. It loads raw prices and auxiliary data, constructs adjusted close/open returns, freezes the annual top-1000 universe, applies point-in-time earnings/short-interest controls, engineers features, applies trading filters, and builds daily cross-sectional ranks.


In [4]:
t0 = time.perf_counter()
prepared = prepare_data(config)
raw_panel = prepared.panel

research_panel = add_point_in_time_features(raw_panel)
research_panel = apply_eligibility_filters(research_panel, config)
research_panel, rank_columns = add_cross_sectional_ranks(research_panel)
model_panel = slim_panel(research_panel, rank_columns)

panel_elapsed = time.perf_counter() - t0
panel_summary = pd.Series({
    'stock_days_raw_panel': len(raw_panel),
    'stock_days_model_panel': len(model_panel),
    'date_min': str(pd.to_datetime(model_panel['date']).min().date()),
    'date_max': str(pd.to_datetime(model_panel['date']).max().date()),
    'ranked_features': len(rank_columns),
    'trade_eligible_stock_days': int(model_panel['is_trade_eligible'].sum()),
    'elapsed_seconds': round(panel_elapsed, 2),
})
panel_summary


stock_days_raw_panel            5470423
stock_days_model_panel          5470423
date_min                     2010-01-04
date_max                     2024-12-31
ranked_features                      28
trade_eligible_stock_days       3194145
elapsed_seconds               64.070000
dtype: object

## 3. Data And Return Sanity Checks

The return identity is checked from the raw price-derived panel. The equal-weight decomposition is also recomputed here rather than read from `outputs/equal_weight_return_decomposition.csv`.


In [5]:
prepared.reconciliation_summary


,tolerance,stock_days_checked,fail_count,fail_fraction,median_abs_residual,p99_abs_residual,max_abs_residual
0,0.000000,5469968,0,0.000000,0.000000,0.000000,0.000000


In [6]:
def annual_stats(series: pd.Series) -> dict[str, float]:
    s = series.dropna()
    return {
        'annual_return': (1.0 + s).prod() ** (252.0 / len(s)) - 1.0,
        'annual_vol': s.std(ddof=1) * np.sqrt(252.0),
        'sharpe': s.mean() / s.std(ddof=1) * np.sqrt(252.0),
        'cumulative_return': (1.0 + s).prod() - 1.0,
        'days': len(s),
    }

return_decomposition = compute_equal_weight_decomposition(raw_panel)
return_summary = pd.DataFrame({
    name: annual_stats(return_decomposition[name])
    for name in ['overnight', 'intraday', 'close_to_close']
}).T
return_summary


,annual_return,annual_vol,sharpe,cumulative_return,days
overnight,0.096600,0.117525,0.843923,2.978977,"3,774.000000"
intraday,0.045526,0.147759,0.375226,0.947889,"3,774.000000"
close_to_close,0.143707,0.194294,0.788834,6.470237,"3,774.000000"


In [7]:
universe_summary = prepared.universe_summary.copy()
eligibility_summary = build_eligibility_summary(research_panel)

print('Universe count by year:')
display(universe_summary[['year', 'universe_count', 'mid_year_exits']].head(3))
display(universe_summary[['year', 'universe_count', 'mid_year_exits']].tail(3))

print('Top eligibility reasons by stock-day count:')
display(
    eligibility_summary.groupby('eligibility_reason', as_index=False)['stock_days']
    .sum()
    .sort_values('stock_days', ascending=False)
)


Universe count by year:


,year,universe_count,mid_year_exits
0,2010,986,0
1,2011,986,0
2,2012,988,0


,year,universe_count,mid_year_exits
12,2022,988,0
13,2023,988,0
14,2024,989,0


Top eligibility reasons by stock-day count:


,eligibility_reason,stock_days
4,OK,3194145
3,MCAP_FAIL,1596321
0,ADV_FAIL,513267
2,EARN_WINDOW,116925
5,PRICE_FAIL,26307
6,VOL_FAIL,22458
1,DATA_FAIL,1000


## 4. Baseline Model Run

The baseline uses the original cross-sectional ranked overnight target, a transparent walk-forward feature-weight model, top/bottom 3% baskets, equal weighting, 5% ADV20 participation cap, and the explicit commission/slippage/borrow cost schedule.


In [8]:
baseline_spec = ExperimentSpec(
    'A_basket_size',
    'top_bottom_3pct',
    notes='Baseline rank-target model run inside this notebook.',
)

t0 = time.perf_counter()
baseline_scored = score_panel_with_target(model_panel, rank_columns, config, 'rank')
baseline_scored = add_walk_forward_alpha_return(baseline_scored, config)
baseline_daily = run_experiment_backtest(baseline_scored, AUM, config, baseline_spec)
baseline_ic = ic_summary(baseline_scored)
baseline_result = summarise_experiment(baseline_daily, baseline_spec, AUM, baseline_ic)
baseline_result['elapsed_seconds'] = round(time.perf_counter() - t0, 2)

pd.Series(baseline_result)[[
    'net_annual_return', 'net_vol', 'net_sharpe', 'max_drawdown',
    'gross_sharpe', 'commission_drag', 'slippage_drag', 'borrow_drag',
    'average_gross_exposure_used', 'average_daily_turnover', 'IC_mean', 'IC_tstat',
    'elapsed_seconds',
]]


  scoring rank target for year 2010...


  scoring rank target for year 2011...


  scoring rank target for year 2012...


  scoring rank target for year 2013...


  scoring rank target for year 2014...


...

  scoring rank target for year 2021...


  scoring rank target for year 2022...


  scoring rank target for year 2023...


  scoring rank target for year 2024...


net_annual_return              0.016869
net_vol                        0.047099
net_sharpe                     0.378713
max_drawdown                  -0.245771
gross_sharpe                   1.728596
commission_drag                0.308959
slippage_drag                  0.928469
borrow_drag                    0.112455
average_gross_exposure_used    0.578925
average_daily_turnover         1.157850
IC_mean                        0.027312
IC_tstat                       8.362962
elapsed_seconds               28.710000
dtype: object

## 5. Promoted Phase 2 Model Run

The promoted final strategy is trained here, from the same point-in-time feature panel. It changes the target to volatility-scaled next overnight return and uses expanding-window transparent feature-weight estimation.


In [9]:
phase2_final_spec = final_phase2_spec()

t0 = time.perf_counter()
final_scored = score_panel_phase2(
    model_panel,
    rank_columns,
    config,
    phase2_final_spec.training_window,
    phase2_final_spec.alpha_learning_method,
)
final_daily, final_positions = final_backtest_with_positions(final_scored, AUM, config)
final_ic = ic_summary(final_scored)
final_result = performance_row(final_daily, AUM)
final_result['IC_mean'], final_result['IC_tstat'] = final_ic
final_result['elapsed_seconds'] = round(time.perf_counter() - t0, 2)

pd.Series(final_result)[[
    'net_annual_return', 'net_vol', 'net_sharpe', 'max_drawdown',
    'gross_sharpe', 'commission_drag', 'slippage_drag', 'borrow_drag',
    'average_gross_exposure_used', 'average_turnover', 'IC_mean', 'IC_tstat',
    'elapsed_seconds',
]]


  Phase 2 scoring expanding / prior50_learned50 for 2010...


  Phase 2 scoring expanding / prior50_learned50 for 2011...


  Phase 2 scoring expanding / prior50_learned50 for 2012...


  Phase 2 scoring expanding / prior50_learned50 for 2013...


  Phase 2 scoring expanding / prior50_learned50 for 2014...


...

  Phase 2 scoring expanding / prior50_learned50 for 2021...


  Phase 2 scoring expanding / prior50_learned50 for 2022...


  Phase 2 scoring expanding / prior50_learned50 for 2023...


  Phase 2 scoring expanding / prior50_learned50 for 2024...


net_annual_return              0.073099
net_vol                        0.049673
net_sharpe                     1.445318
max_drawdown                  -0.101865
gross_sharpe                   3.111879
commission_drag                0.377699
slippage_drag                  1.136577
borrow_drag                    0.152285
average_gross_exposure_used    0.749713
average_turnover               1.499426
IC_mean                        0.028940
IC_tstat                       9.934090
elapsed_seconds               47.910000
dtype: object

In [10]:
model_comparison = pd.DataFrame([
    {
        'model': 'baseline_rank_target_4y',
        'target': 'cross_sectional_rank(overnight_next)',
        'training_window': f'{config.training_years}y rolling',
        'net_annual_return': baseline_result['net_annual_return'],
        'net_vol': baseline_result['net_vol'],
        'net_sharpe': baseline_result['net_sharpe'],
        'max_drawdown': baseline_result['max_drawdown'],
        'IC_mean': baseline_result['IC_mean'],
        'IC_tstat': baseline_result['IC_tstat'],
    },
    {
        'model': phase2_final_spec.experiment_id,
        'target': 'overnight_next / (vol20 / sqrt(252))',
        'training_window': phase2_final_spec.training_window,
        'net_annual_return': final_result['net_annual_return'],
        'net_vol': final_result['net_vol'],
        'net_sharpe': final_result['net_sharpe'],
        'max_drawdown': final_result['max_drawdown'],
        'IC_mean': final_result['IC_mean'],
        'IC_tstat': final_result['IC_tstat'],
    },
])
model_comparison


,model,target,training_window,net_annual_return,net_vol,net_sharpe,max_drawdown,IC_mean,IC_tstat
0,baseline_rank_target_4y,cross_sectional_rank(overnight_next),4y rolling,0.016869,0.047099,0.378713,-0.245771,0.027312,8.362962
1,phase2_g5_05_expanding,overnight_next / (vol20 / sqrt(252)),expanding,0.073099,0.049673,1.445318,-0.101865,0.028940,9.934090


## 6. Validation, Holdout, And Stress Windows

The final model is split into design, validation, internal holdout, and full development periods. These rows are computed from the freshly generated final daily returns and final scored panel.


In [11]:
period_rows = []
for period_name, start, end in PERIODS:
    period_rows.append(
        summarise_period(
            final_daily,
            phase2_final_spec,
            AUM,
            period_name,
            start,
            end,
            ic_for_period(final_scored, start, end),
        )
    )
period_summary = pd.DataFrame(period_rows)
period_summary[[
    'period', 'start_date', 'end_date', 'net_annual_return', 'net_vol',
    'net_sharpe', 'max_drawdown', 'IC_mean', 'IC_tstat', 'worst_12m_return',
    'average_gross_exposure_used', 'fraction_cap_binding_days',
]]


,period,start_date,end_date,net_annual_return,net_vol,net_sharpe,max_drawdown,IC_mean,IC_tstat,worst_12m_return,average_gross_exposure_used,fraction_cap_binding_days
0,design_2010_2018,2010-01-01,2018-12-31,0.039625,0.033697,1.170126,-0.101865,0.035559,9.630745,-0.100903,0.628556,1.000000
1,validation_2019_2022,2019-01-01,2022-12-31,0.174141,0.071589,2.278993,-0.054656,0.019862,3.336067,0.065084,0.911750,1.000000
2,internal_holdout_2023_2024,2023-01-01,2024-12-31,0.033329,0.055327,0.620163,-0.076432,0.017290,2.256992,-0.075479,0.970761,0.998008
3,full_2010_2024,2010-01-01,2024-12-31,0.073099,0.049673,1.445318,-0.101865,0.028940,9.934090,-0.100903,0.749713,0.999735


In [12]:
stress_windows = {
    'late_2018': ('2018-10-01', '2018-12-31'),
    'covid_q1_2020': ('2020-01-01', '2020-03-31'),
    'drawdown_2022': ('2022-01-01', '2022-12-31'),
}

stress_rows = []
final_daily_for_stress = final_daily.copy()
final_daily_for_stress['date'] = pd.to_datetime(final_daily_for_stress['date'])
for name, (start, end) in stress_windows.items():
    chunk = final_daily_for_stress.loc[final_daily_for_stress['date'].between(start, end)]
    stress_rows.append({
        'window': name,
        'start': start,
        'end': end,
        'net_return': (1.0 + chunk['net_return']).prod() - 1.0,
        'net_sharpe': chunk['net_return'].mean() / chunk['net_return'].std(ddof=1) * np.sqrt(252.0),
        'max_drawdown': ((1.0 + chunk['net_return']).cumprod() / (1.0 + chunk['net_return']).cumprod().cummax() - 1.0).min(),
        'avg_gross_exposure': chunk['gross_exposure'].mean(),
        'days': len(chunk),
    })
pd.DataFrame(stress_rows)


,window,start,end,net_return,net_sharpe,max_drawdown,avg_gross_exposure,days
0,late_2018,2018-10-01,2018-12-31,-0.005931,-0.482211,-0.019200,0.852784,63
1,covid_q1_2020,2020-01-01,2020-03-31,0.075478,2.798751,-0.041378,0.889505,62
2,drawdown_2022,2022-01-01,2022-12-31,0.320946,3.524723,-0.026347,0.992267,251


## 7. Feature Ablation And Model Risk

This section reruns selected final-model variants inside the notebook. The full model is compared with two high-signal ablations: removing return/reversal features and removing short-interest/borrow-stress features.


In [13]:
ablation_variants = [
    ('full_model', 'none', rank_columns),
    ('remove_return_reversal', 'return/reversal', [c for c in rank_columns if c not in FEATURE_GROUPS['return/reversal']]),
    ('remove_short_interest_borrow_stress', 'short-interest/borrow-stress', [c for c in rank_columns if c not in FEATURE_GROUPS['short-interest/borrow-stress']]),
]

ablation_rows = []
for variant_name, removed_group, columns in ablation_variants:
    t0 = time.perf_counter()
    if variant_name == 'full_model':
        scored_variant = final_scored
        daily_variant = final_daily
    else:
        scored_variant = score_panel_phase2(
            model_panel,
            columns,
            config,
            phase2_final_spec.training_window,
            phase2_final_spec.alpha_learning_method,
        )
        daily_variant = run_phase2_backtest(scored_variant, AUM, config, phase2_final_spec)
    row = summarise_period(
        daily_variant,
        phase2_final_spec,
        AUM,
        'full_2010_2024',
        '2010-01-01',
        '2024-12-31',
        ic_summary(scored_variant),
    )
    row['variant'] = variant_name
    row['removed_group'] = removed_group
    row['feature_count'] = len(columns)
    row['elapsed_seconds'] = round(time.perf_counter() - t0, 2)
    ablation_rows.append(row)

ablation_summary = pd.DataFrame(ablation_rows)
ablation_summary[[
    'variant', 'removed_group', 'feature_count', 'net_annual_return', 'net_vol',
    'net_sharpe', 'max_drawdown', 'IC_mean', 'IC_tstat', 'average_gross_exposure_used',
    'elapsed_seconds',
]]


  Phase 2 scoring expanding / prior50_learned50 for 2010...


  Phase 2 scoring expanding / prior50_learned50 for 2011...


  Phase 2 scoring expanding / prior50_learned50 for 2012...


  Phase 2 scoring expanding / prior50_learned50 for 2013...


  Phase 2 scoring expanding / prior50_learned50 for 2014...


...

  Phase 2 scoring expanding / prior50_learned50 for 2021...


  Phase 2 scoring expanding / prior50_learned50 for 2022...


  Phase 2 scoring expanding / prior50_learned50 for 2023...


  Phase 2 scoring expanding / prior50_learned50 for 2024...


,variant,removed_group,feature_count,net_annual_return,net_vol,net_sharpe,max_drawdown,IC_mean,IC_tstat,average_gross_exposure_used,elapsed_seconds
0,full_model,none,28,0.073099,0.049673,1.445318,-0.101865,0.028940,9.934090,0.749713,1.890000
1,remove_return_reversal,return/reversal,22,-0.036228,0.032647,-1.113876,-0.466622,0.011262,3.332858,0.476798,32.440000
2,remove_short_interest_borrow_stress,short-interest/borrow-stress,25,0.076977,0.050010,1.508078,-0.113176,0.028908,9.846413,0.783239,34.060000


## 8. Capacity, Cost, And Position Checks

The final 250M portfolio is generated in this notebook. These checks use the fresh positions and daily returns, not the submitted parquet file.


In [14]:
capacity_cost_summary = pd.Series({
    'days': len(final_daily),
    'positions': len(final_positions),
    'average_gross_exposure': final_daily['gross_exposure'].mean(),
    'fraction_cap_binding_days': final_daily['cap_binding'].mean(),
    'average_turnover': final_daily['turnover'].mean(),
    'average_long_names': final_daily['n_long'].mean(),
    'average_short_names': final_daily['n_short'].mean(),
    'avg_commission_bps_per_day': final_daily['commission_cost'].mean() * 10_000,
    'avg_slippage_bps_per_day': final_daily['slippage_cost'].mean() * 10_000,
    'avg_borrow_bps_per_day': final_daily['borrow_cost'].mean() * 10_000,
    'max_participation_rate': final_positions['participation_rate'].max(),
})
capacity_cost_summary


days                           3,774.000000
positions                    188,070.000000
average_gross_exposure             0.749713
fraction_cap_binding_days          0.999735
average_turnover                   1.499426
average_long_names                24.916534
average_short_names               24.916534
avg_commission_bps_per_day         0.749713
avg_slippage_bps_per_day           2.249139
avg_borrow_bps_per_day             0.300775
max_participation_rate             0.050000
dtype: float64

In [15]:
borrow_tier_summary = (
    final_positions.loc[final_positions['side'].eq('SHORT')]
    .groupby('borrow_tier')
    .agg(
        short_position_days=('borrow_tier', 'size'),
        average_short_notional=('abs_dollar_position', 'mean'),
        total_borrow_cost=('borrow_cost', 'sum'),
    )
    .reset_index()
)
borrow_tier_summary


,borrow_tier,short_position_days,average_short_notional,total_borrow_cost
0,A,40629,"4,352,833.192034","2,807,162.853320"
1,B,35972,"3,568,609.022803","10,188,095.537163"
2,C,17434,"2,779,389.714057","15,382,819.134877"


## 9. Submitted Artefact Audit

Only now do we read submitted output files. The purpose is to verify that the generated-in-notebook final model matches the included submission artefacts.


In [16]:
submitted_perf = pd.read_csv(config.output_dir / 'performance_summary.csv')
submitted_daily = pd.read_csv(config.output_dir / 'daily_returns_250m.csv')
submitted_daily['date'] = pd.to_datetime(submitted_daily['date'])
submitted_positions = pd.read_parquet(config.output_dir / 'positions_250m.parquet')

fresh_daily = final_daily.copy()
fresh_daily['date'] = pd.to_datetime(fresh_daily['date'])
joined_daily = fresh_daily[['date', 'net_return']].merge(
    submitted_daily[['date', 'net_return']],
    on='date',
    suffixes=('_fresh', '_submitted'),
    how='outer',
)
joined_daily['abs_diff'] = (joined_daily['net_return_fresh'] - joined_daily['net_return_submitted']).abs()

submitted_row = submitted_perf.loc[submitted_perf['AUM'].eq(AUM)].iloc[0]
audit = pd.Series({
    'submitted_perf_rows': len(submitted_perf),
    'submitted_daily_rows': len(submitted_daily),
    'fresh_daily_rows': len(fresh_daily),
    'max_abs_daily_net_return_diff': joined_daily['abs_diff'].max(),
    'fresh_net_sharpe': final_result['net_sharpe'],
    'submitted_net_sharpe': submitted_row['net_sharpe'],
    'fresh_positions': len(final_positions),
    'submitted_positions': len(submitted_positions),
})
assert audit['max_abs_daily_net_return_diff'] < 1e-12
assert abs(audit['fresh_net_sharpe'] - audit['submitted_net_sharpe']) < 1e-12
audit


submitted_perf_rows                   3.000000
submitted_daily_rows              3,774.000000
fresh_daily_rows                  3,774.000000
max_abs_daily_net_return_diff         0.000000
fresh_net_sharpe                      1.445318
submitted_net_sharpe                  1.445318
fresh_positions                 188,070.000000
submitted_positions             188,070.000000
dtype: float64

## 10. Submitted Tests

The notebook finishes by running the submitted tests directly with `PYTHONPATH=src`.


In [17]:
env = os.environ.copy()
env['PYTHONPATH'] = str(SRC)
result = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q'],
    cwd=ROOT,
    env=env,
    text=True,
    capture_output=True,
)
print(result.stdout)
if result.stderr.strip():
    print(result.stderr)
assert result.returncode == 0


.............                                                            [100%]
13 passed in 0.83s



## Research Conclusion

The notebook-run evidence supports the promoted final strategy over the baseline inside the 2010-2024 development window. The main material risk remains feature concentration: the return/reversal group is highly important, so the final report should treat it as a model-dependence risk rather than a universally robust alpha claim.
